# Seance 9 — Bivariee II : barres groupees/empilees, boxplots croises, scatter par groupe

Voir user guide :
- Marks (3/3) : https://altair-viz.github.io/user_guide/marks/index.html

## Objectifs
- Comparer des groupes (education, ideologie)
- Comparer des distributions (boxplot)
- Construire des barres **groupees** et **empilees** a partir de tableaux pandas

In [ ]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

data_url = "https://raw.githubusercontent.com/datamisc/ts-2024/main/data.csv"
df = pd.read_csv(data_url, compression="gzip", low_memory=False)

## 1) Vote presidentiel (post) selon l'education

Variable : `V242096x` (summary : vote presidentiel 2024)
- 1 = Harris
- 2 = Trump
- 3-6 = autres

Variable de groupe : `V241200` (education 1-5).

On commence par construire un tableau en **forme longue** avec pandas.

In [ ]:
tmp = df.loc[(df['V241200'] > 0) & (df['V242096x'] > 0), ['V241200', 'V242096x']].copy()
tmp['education'] = tmp['V241200'].astype(int)
tmp['vote'] = tmp['V242096x'].astype(int)
tmp['vote_label'] = tmp['vote'].replace({
    1: 'Harris',
    2: 'Trump',
    3: 'Autres',
    4: 'Autres',
    5: 'Autres',
    6: 'Autres',
})

tmp[['education','vote_label']].head()

### 1a) Barres empilees (proportions)

On calcule la proportion de chaque choix de vote **dans chaque niveau d'education**.

In [ ]:
tab_prop = pd.crosstab(tmp['education'], tmp['vote_label'], normalize='index').reset_index()
tab_prop_long = tab_prop.melt(id_vars='education', var_name='vote_label', value_name='proportion')

tab_prop_long.head()

In [ ]:
alt.Chart(tab_prop_long).mark_bar().encode(
    x=alt.X('education', type='ordinal', title='Education (code 1-5)'),
    y=alt.Y('proportion', type='quantitative', title='Proportion', axis=alt.Axis(format='%')),
    color=alt.Color('vote_label', type='nominal', title='Vote'),
    order=alt.Order('vote_label', type='nominal')
).properties(
    title=alt.TitleParams(
        text="Vote presidentiel (proportions) selon l'education",
        subtitle=[
            "Les proportions de Harris/Trump varient selon l'education.",
            "Source : ANES 2024 Time Series Study (V242096x)"
        ],
        anchor='start'
    ),
    width=600,
    height=320
)

### Hack-Time 1 (10 min)

Refaites le meme graphique mais en remplaçant l'education par l'ideologie (`V241177`, 1-7).
- filtrez 1-7
- recodez ou gardez les codes
- calculez des proportions avec `pd.crosstab(..., normalize='index')`

In [ ]:
# Hack-Time 1 : votre code ici

### 1b) Barres groupees (comptages)

On calcule les comptages par education et vote, puis on utilise `xOffset` pour afficher des barres cote a cote.

In [ ]:
tab_n = pd.crosstab(tmp['education'], tmp['vote_label']).reset_index()
tab_n_long = tab_n.melt(id_vars='education', var_name='vote_label', value_name='n')

alt.Chart(tab_n_long).mark_bar().encode(
    x=alt.X('education', type='ordinal', title='Education (code 1-5)'),
    xOffset=alt.XOffset('vote_label', type='nominal'),
    y=alt.Y('n', type='quantitative', title='Nombre'),
    color=alt.Color('vote_label', type='nominal', title='Vote')
).properties(
    title=alt.TitleParams(
        text="Vote presidentiel (comptages) selon l'education",
        subtitle=[
            "Attention : les comptages dependent aussi de la taille des groupes.",
            "Source : ANES 2024 Time Series Study (V242096x)"
        ],
        anchor='start'
    ),
    width=650,
    height=320
)

## 2) Boxplot croise (pandas) : evaluation de Harris selon le vote

Question : les personnes qui votent Trump evaluent-elles Harris tres différemment ?

Variables :
- `V242096x` (vote) -> on garde Harris vs Trump
- `V241156` (thermometre Harris, 0-100)

On calcule q1/q3/median/min/max par groupe avec pandas, puis on dessine le boxplot par groupe.

In [ ]:
df_box = df.loc[
    df['V242096x'].isin([1, 2]) & df['V241156'].between(0, 100),
    ['V242096x', 'V241156']
].copy()

df_box['vote_label'] = df_box['V242096x'].replace({1: 'Harris', 2: 'Trump'})

stats = (
    df_box.groupby('vote_label')['V241156']
    .agg(
        q1=lambda s: s.quantile(0.25),
        q3=lambda s: s.quantile(0.75),
        median='median',
        min='min',
        max='max'
    )
    .reset_index()
)

stats

In [ ]:
box_rect = alt.Chart(stats).mark_bar(size=30, color='#60A5FA').encode(
    x=alt.X('vote_label', type='nominal', title='Vote (post)'),
    y=alt.Y('q1', type='quantitative', title='Thermometre Harris (0-100)', scale=alt.Scale(domain=[0, 100])),
    y2=alt.Y2('q3', type='quantitative')
)

whisker = alt.Chart(stats).mark_rule(color='black').encode(
    x=alt.X('vote_label', type='nominal'),
    y=alt.Y('min', type='quantitative'),
    y2=alt.Y2('max', type='quantitative')
)

med = alt.Chart(stats).mark_rule(color='white', strokeWidth=3).encode(
    x=alt.X('vote_label', type='nominal'),
    y=alt.Y('median', type='quantitative')
)

alt.layer(whisker, box_rect, med).properties(
    title=alt.TitleParams(
        text="Evaluation de Harris selon le vote (post)",
        subtitle=[
            "Les electeurs Trump donnent des notes beaucoup plus basses a Harris.",
            "Source : ANES 2024 Time Series Study (V242096x, V241156)"
        ],
        anchor='start'
    ),
    width=520,
    height=360
)

### Hack-Time 2 (15 min)

Reproduisez le boxplot croise pour :
- `V241157` (thermometre Trump)
- selon `V242096x` (Harris vs Trump)

Indication : changez la variable dans `df_box` et dans l'agregation pandas.

In [ ]:
# Hack-Time 2 : votre code ici

## 3) Scatter par groupe : polarisation selon le vote

On reprend la relation Harris vs Trump, mais on colore selon le vote post (Harris/Trump).

In [ ]:
df_sc = df.loc[
    df['V242096x'].isin([1, 2]) & df['V241156'].between(0, 100) & df['V241157'].between(0, 100),
    ['V242096x', 'V241156', 'V241157']
].copy()
df_sc['vote_label'] = df_sc['V242096x'].replace({1: 'Harris', 2: 'Trump'})

alt.Chart(df_sc.sample(2000, random_state=2)).mark_circle(size=25, opacity=0.35).encode(
    x=alt.X('V241157', type='quantitative', title='Thermometre Trump', scale=alt.Scale(domain=[0, 100])),
    y=alt.Y('V241156', type='quantitative', title='Thermometre Harris', scale=alt.Scale(domain=[0, 100])),
    color=alt.Color('vote_label', type='nominal', title='Vote (post)')
).properties(
    title=alt.TitleParams(
        text="Polarisation affective selon le vote",
        subtitle=[
            "Les electeurs Harris sont plutot en haut-gauche, les electeurs Trump en bas-droite.",
            "Source : ANES 2024 Time Series Study"
        ],
        anchor='start'
    ),
    width=520,
    height=420
)

### Hack-Time 3 (10 min)

Changez la couleur pour une autre variable de statut :
- education (`V241200`)
- revenu (`V242025`)

Astuce : filtrez les valeurs invalides (education > 0, revenu >= 0).

In [ ]:
# Hack-Time 3 : votre code ici